# 🚀 Tutorial 4: Autonomous Autopilot FSM & Blackbox Telemetry

In this tutorial, you will:
1. Initialize the **Autonomous Supervisory Agent**.
2. Step through the 5-State Finite State Machine (FSM) closed-loop decision cycle:
   $$\text{Sense} \rightarrow \text{Infer} \rightarrow \text{Diagnose} \rightarrow \text{Optimize} \rightarrow \text{Actuate} \rightarrow \text{Maintain}$$
3. Execute a fast multi-phase autonomous stress test mission.
4. Parse and inspect the **Blackbox Flight Recorder log** (`reports/autonomous_flight_log.json`).

In [ ]:
import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.autonomous.autopilot import AutonomousSupervisoryAgent, AutonomousFSMState
from src.autonomous.stress_test import AutonomousStressTestRunner

print("[*] Autonomous platform initialized.")

## 1. Interactive Step-by-Step Decision Loop

In [ ]:
agent = AutonomousSupervisoryAgent(dt_sec=2.0)
agent.reset()

# Step 1: Cold Startup
st, cmd = agent.step(mission_phase="COLD_STARTUP", moisture_override=12.0)
print(f"[Step 1] FSM State: {cmd.fsm_state} | Temp: {st.reactor_temp_c:.1f}°C | Burner: {cmd.burner_duty_pct:.0f}%")
print(f"         Action: {cmd.action_summary}\n")

# Fast-forward reactor temperature to steady state
agent.current_temp_c = 495.0
st, cmd = agent.step(mission_phase="NOMINAL_CRUISE", moisture_override=12.0)
print(f"[Step 2] FSM State: {cmd.fsm_state} | Temp: {st.reactor_temp_c:.1f}°C | Feed: {st.feed_rate_kg_h:.1f} kg/h | TSI: {st.tsi_pct:.1f}%")
print(f"         Action: {cmd.action_summary}\n")

# Step 3: Inject Cyclone Blockage Fault
st, cmd = agent.step(mission_phase="FAULT_INJECTION", injected_fault="cyclone_blockage")
print(f"[Step 3] FSM State: {cmd.fsm_state} | Pulse-Jet Active: {cmd.pulse_jet_active} | Blowback Duration: {cmd.pulse_jet_duration_sec}s")
print(f"         Action: {cmd.action_summary}")

## 2. Executing Autonomous Stress Test & Flight Log Analysis

In [ ]:
runner = AutonomousStressTestRunner(dt_sec=10.0)
summary = runner.run_4hour_mission()

print(f"=== Mission Overall Status: {summary['overall_status']} ===")
print(f"Phases Completed : {summary['phases_executed_count']} / 6")
print(f"Total Flight Time: {summary['total_flight_time_hours']} hours ({summary['total_flight_time_minutes']} minutes)\n")

print("Mission Timeline Breakdown:")
for p in summary["phases"]:
    print(f"  {p['phase_name']:<38} [{p['start_time_min']:>5.1f} - {p['end_time_min']:>5.1f} min] Temp: {p['initial_temp_c']:>5.1f}->{p['final_temp_c']:>5.1f}°C ({p['final_fsm_state']})")

# Inspect blackbox flight recorder log
log_file = ROOT_DIR / "reports" / "autonomous_flight_log.json"
if log_file.is_file():
    with open(log_file, "r") as f:
        log_data = json.load(f)
    print(f"\n[*] Blackbox flight log verified: {log_data['flight_recorder_summary']['total_events_logged']} discrete safety events recorded.")